# Balanço Hídrico com dados da Open-Meteo (Google Colab)

Este notebook usa `main.py` (busca de dados na Open-Meteo), `Calcula_ETo.py` (cálculo do ETo) e `Balanco_Hidrico.py` (simulação do balanço hídrico), do repositório [`ManejoClima`](https://github.com/Hidrovales/ManejoClima), para:

1. Buscar as coordenadas do local;
2. Registrar o dia de hoje na série acumulada (rodar uma vez por dia);
3. Acompanhar a série que vai crescendo com o tempo;
4. Rodar o balanço hídrico com o que já foi acumulado;
5. **Visualizar os gráficos** gerados por `Balanco_Hidrico.py`.

> Rode a célula de **registrar o dia de hoje** uma vez por dia — é isso que alimenta a série aos poucos.

> O banco de dados é salvo no seu **Google Drive**, para persistir entre sessões (o disco do Colab é apagado quando a sessão reinicia).

## Preparação: clonar o repositório

In [ ]:
import os

REPO_URL = "https://github.com/Hidrovales/ManejoClima.git"
REPO_DIR = "/content/ManejoClima"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
%matplotlib inline

import main
import Calcula_ETo as gse
import Balanco_Hidrico as bh

## Preparação: montar o Google Drive

Na primeira vez vai pedir autorização de acesso ao seu Drive. O banco fica salvo em `MyDrive/ManejoClima/balanco_hidrico.db`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PASTA_DADOS = "/content/drive/MyDrive/ManejoClima"
os.makedirs(PASTA_DADOS, exist_ok=True)
DATABASE_PATH = f"{PASTA_DADOS}/balanco_hidrico.db"
DATABASE_PATH

## 1. Buscar coordenadas do local

In [ ]:
NOME_CIDADE = "Rio Pardo de Minas, Brasil"

local = main.buscar_coordenadas(NOME_CIDADE)
local

## 2. Registrar o dia de hoje

Busca os dados diários de hoje na Open-Meteo, calcula o ETo e grava (ou atualiza, se já rodou hoje) uma linha na série acumulada. **Rode essa célula uma vez por dia** para a série crescer.

In [ ]:
resultado_hoje = bh.registrar_dia(
    local=local["nome"],
    latitude=local["latitude"],
    longitude=local["longitude"],
    altitude=local["altitude"],
    database_path=DATABASE_PATH,
    timezone=local.get("timezone", "auto"),
)
resultado_hoje

## 3. Ver a série acumulada até agora

In [ ]:
serie = bh.carregar_serie_diaria(local["nome"], DATABASE_PATH)
serie

## 4. Parâmetros da cultura e balanço hídrico

Ajuste os valores abaixo para a sua cultura. Se a série ainda não cobrir o ciclo inteiro, `balanco()` avisa e calcula até onde já há dado — sem quebrar.

In [ ]:
CULTURA = "Milho"

periodo = {"inicial": 15, "desenvolvimento": 30, "media": 60, "final": 15}  # dias de cada fase
kc_etapas = {"inicial": 0.5, "media": 1.2, "final": 0.8}
forma_kc = {"inicial": True, "desenvolvimento": False, "media": True, "final": False}
z_etapas = {"inicial": 0.15, "media": 0.40, "final": 0.30}   # profundidade radicular [m]
forma_z = {"inicial": True, "desenvolvimento": False, "media": True, "final": False}

theta_fc = 0.23   # capacidade de campo [m3/m3]
theta_wp = 0.10   # ponto de murcha [m3/m3]
p = 0.5           # fator de disponibilidade hídrica

data_in = {"ano": 2026, "mes": 9, "dia": 25}  # data de início do cultivo

eto_df = serie[["DATA", "ETO"]]
p_df = serie[["DATA", "PRECIPITACAO"]]

bh.balanco(
    local=local["nome"], cultura=CULTURA,
    theta_fc=theta_fc, theta_wp=theta_wp, p=p,
    P=p_df, eto=eto_df,
    periodo=periodo, z_etapas=z_etapas, forma_z=forma_z,
    kc_etapas=kc_etapas, forma_kc=forma_kc,
    data_in=data_in, database_path=DATABASE_PATH,
)

## 5. Visualizar os resultados

In [ ]:
resultado = bh.carregar_resultado(local["nome"], CULTURA, DATABASE_PATH)
bh.plot_extras(resultado, figsize=(9, 4))

In [ ]:
bh.plot_balanco(resultado, figsize=(9, 4))